# Detecting Fraudulent Mobile-Money Transactions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS-Shahad/Fraud_Detection_Model/blob/main/notebooks/fraud_detection.ipynb)

**Goal:** flag fraudulent transactions in the [PaySim](https://www.kaggle.com/datasets/ealtaf/paysim1) dataset,
a simulation of 6.3 million mobile-money transactions built from real logs of a mobile-money service.

**Why it's hard:** only about 0.13% of transactions are fraud. A model that calls everything legitimate is
99.87% accurate and completely useless, so this notebook judges models on **precision, recall and PR-AUC
for the fraud class**, not on accuracy.

**Contents**
1. Setup
2. Load the data
3. Exploratory analysis
4. Feature engineering
5. Train / validation / test split
6. Model training and comparison
7. Results on the test set
8. What drives the model
9. Save the model
10. Conclusions and next steps

## 1. Setup

The reusable code (feature engineering, models, metrics, charts) lives in the `src/fraud_detection` package
so the notebook and the command-line training script share exactly the same logic.
When run in Google Colab, the cell below clones the repository and installs it.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !git clone -q https://github.com/CS-Shahad/Fraud_Detection_Model.git
    %cd Fraud_Detection_Model
    !pip install -q -e .
    sys.path.insert(0, "src")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from fraud_detection import plots
from fraud_detection.config import FIGURES_DIR, TARGET
from fraud_detection.data import find_data_file, load_transactions
from fraud_detection.evaluation import classification_metrics, comparison_table
from fraud_detection.features import FEATURES, FRAUD_PRONE_TYPES, add_features, filter_fraud_prone
from fraud_detection.train import best_tree_model_name, feature_importances, save_outputs, train_and_evaluate

pd.set_option("display.float_format", "{:,.4f}".format)
plots.apply_style()

## 2. Load the data

`find_data_file` uses `data/PS_20174392719_1491204439457_log.csv` if it exists and otherwise downloads the
dataset from Kaggle. To use a copy in Google Drive instead, mount the drive and set `DATA_PATH` to the file.

In [ ]:
DATA_PATH = None  # e.g. "/content/drive/MyDrive/Fraud_data/PS_20174392719_1491204439457_log.csv"

df = load_transactions(find_data_file(DATA_PATH))
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

| Column | Meaning |
|---|---|
| `step` | Hour since the start of the simulation (1 to 743, about 31 days) |
| `type` | PAYMENT, TRANSFER, CASH_OUT, DEBIT or CASH_IN |
| `amount` | Transaction amount |
| `nameOrig` / `nameDest` | Sender and receiver IDs |
| `oldbalanceOrg` / `newbalanceOrig` | Sender balance before and after |
| `oldbalanceDest` / `newbalanceDest` | Receiver balance before and after |
| `isFraud` | **Target:** 1 if the transaction is fraudulent |
| `isFlaggedFraud` | Output of the existing rule: flags transfers above 200,000 |

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
counts = df[TARGET].value_counts()
print(f"Fraudulent: {counts[1]:,} ({counts[1] / len(df):.3%})")
print(f"Legitimate: {counts[0]:,}")

## 3. Exploratory analysis

### Which transaction types carry fraud?

In [ ]:
display(pd.crosstab(df["type"], df[TARGET], margins=True).rename(columns={0: "legitimate", 1: "fraud"}))
plots.fraud_rate_by_type(df, path=FIGURES_DIR / "fraud_rate_by_type.png");

Every fraudulent transaction is a **TRANSFER** or a **CASH_OUT**. That matches the fraud pattern the
simulator models: take over an account, transfer the money to a mule account, then cash it out.
PAYMENT, DEBIT and CASH_IN can therefore be marked legitimate by rule, which removes over half the rows
before modelling.

### Do fraudulent amounts look different?

In [ ]:
prone = filter_fraud_prone(df)
plots.amount_distribution(prone, path=FIGURES_DIR / "amount_distribution.png");
prone.groupby(TARGET)["amount"].describe()

### When does fraud happen?

In [ ]:
plots.fraud_rate_by_hour(prone, path=FIGURES_DIR / "fraud_rate_by_hour.png");

Legitimate activity follows a daily rhythm, while fraud is spread more evenly through the day, so the
*share* of transactions that are fraud rises in the hours when legitimate traffic is quiet. The hour of day
(`step % 24`) is therefore kept as a feature.

### How good is the existing rule?

In [ ]:
rule = classification_metrics(df[TARGET], df["isFlaggedFraud"], threshold=0.5)
print(f"isFlaggedFraud flags {df['isFlaggedFraud'].sum()} transactions")
print(f"precision {rule['precision']:.2%}, recall {rule['recall']:.2%}")

The current rule is precise but catches only a tiny fraction of fraud. It is the baseline the models need
to beat. It is not used as a feature.

## 4. Feature engineering

The most telling signal in this data is whether an account's balances **add up**. For a legitimate
transfer, the sender's balance falls by the amount and the receiver's balance rises by it. Fraudulent
transactions often break this, for example money arrives but the receiver still shows a zero balance.

| Feature | Definition |
|---|---|
| `errorBalanceOrig` | `newbalanceOrig + amount - oldbalanceOrg` |
| `errorBalanceDest` | `oldbalanceDest + amount - newbalanceDest` |
| `origEmptied` | Sender had money and was left with exactly 0 |
| `destBalancesZero` | Receiver's balance is 0 before and after a non-zero transaction |
| `isTransfer` | TRANSFER (1) or CASH_OUT (0) |
| `hourOfDay` | `step % 24` |

The account IDs are dropped: almost every ID appears only once, so they can't generalise.

In [ ]:
featured = add_features(prone)
featured.groupby(TARGET)[["origEmptied", "destBalancesZero"]].mean().rename(index={0: "legitimate", 1: "fraud"})

## 5. Train / validation / test split

The data is split **60 / 20 / 20** with stratification, so each part keeps the same fraud rate:

- **train**: fit the models
- **validation**: pick each model's decision threshold and choose the best model
- **test**: touched once, at the end, for the final numbers

Choosing thresholds on the test set would make the reported results look better than they would be on new
data.

`train_and_evaluate` does the split, fits each model, tunes its threshold on validation (the cut-off that
maximises F1), and scores the test set.

## 6. Model training and comparison

Four models, from simple to strong:

- **Logistic Regression**: a linear baseline, with log-scaled inputs and balanced class weights
- **Random Forest**: an ensemble of decision trees
- **XGBoost** and **LightGBM**: gradient-boosted trees, usually the strongest choice for tabular data

Instead of oversampling, class imbalance is handled by **tuning the decision threshold** on the validation
set. This keeps the predicted probabilities meaningful.

In [ ]:
result = train_and_evaluate(df)

## 7. Results on the test set

In [ ]:
all_metrics = {"Rule baseline (isFlaggedFraud)": result.baseline_metrics, **result.test_metrics}
table = comparison_table(all_metrics)
table.style.format(
    {c: "{:.4f}" for c in ["pr_auc", "roc_auc", "precision", "recall", "f1"]}
    | {"fraud_amount_caught": "{:.2%}", "fp": "{:,}", "fn": "{:,}"}
)

- **PR-AUC**: area under the precision-recall curve. It summarises performance across all thresholds and,
  unlike ROC-AUC, is not inflated by the huge number of easy legitimate transactions.
- **precision**: of the transactions flagged, the share that really are fraud (a low value means more
  customers get blocked for nothing).
- **recall**: of all fraud, the share that is caught.
- **fraud_amount_caught**: the share of stolen money in the test set that the model would have stopped.
- **fp / fn**: false alarms and missed frauds.

In [ ]:
b = result.baseline_metrics
plots.precision_recall_curves(
    result.y_test, result.test_probas, baseline=(b["recall"], b["precision"], "isFlaggedFraud rule")
);

In [ ]:
best = result.best_model_name
print(f"Best model on validation PR-AUC: {best} (threshold {result.thresholds[best]:.3f})")
plots.confusion_matrix_plot(result.test_metrics[best], f"{best} on the test set");

## 8. What drives the model

In [ ]:
name = best_tree_model_name(result)
plots.feature_importance(feature_importances(result.models[name]), f"What drives the {name} model");

The balance features dominate, which confirms the idea from section 4: fraud in PaySim shows up as
balances that don't add up and sender accounts that get emptied.

## 9. Save the model

`save_outputs` writes:

- `models/fraud_model.joblib`: the best model, its threshold and its feature list
- `reports/metrics.json` and `reports/model_comparison.md`: the test results
- `reports/figures/`: the charts used in the README

`fraud_detection.predict.FraudScorer` loads the model and scores new transactions.

In [ ]:
save_outputs(result)

from fraud_detection.predict import FraudScorer

scorer = FraudScorer()
sample = df.sample(5, random_state=0)
pd.concat([sample[["type", "amount", TARGET]], scorer.score(sample)], axis=1)

## 10. Conclusions and next steps

**Conclusions**

- Fraud in PaySim is confined to TRANSFER and CASH_OUT, and the clearest signal is balances that don't add
  up. Features that measure this make every model much stronger than raw balances alone.
- Gradient-boosted trees and Random Forest far outperform both logistic regression and the existing
  `isFlaggedFraud` rule. See the results table for the exact numbers.
- Choosing the threshold on a validation set, rather than using the default 0.5, controls the trade-off
  between missed fraud and false alarms.

**Limitations**

- PaySim is simulated. The balance-error features are very strong here partly because of how the simulator
  records fraud, and real data is unlikely to separate this cleanly.
- The balances *after* a transaction are only known once it is processed, so this setup detects fraud
  just after the fact, not before authorisation.

**Next steps**

- Validate on a time-based split (train on early days, test on later ones) to measure drift.
- Pick the threshold from a cost model (the cost of a missed fraud versus a false alarm) instead of F1.
- Add per-account behaviour features (for example transaction velocity) and explain individual
  predictions with SHAP.
- Serve `FraudScorer` behind a small REST API.